In [1]:
import bw2analyzer as ba
import bw2data as bd
import bw2calc as bc
import bw2io as bi
import matrix_utils as mu
import bw_processing as bp
import time
import wurst
import os
from pathlib import Path
import copy
import numpy as np
import pandas as pd
import openpyxl
import re 

import warnings
warnings.filterwarnings("ignore")

In [2]:
start_time = time.time()

In [3]:
bd.projects.set_current('nitrous_oxide_production_monte_carlo')

In [4]:
sorted(list(bd.databases))

['ecoinvent-3.10-biosphere', 'ecoinvent-3.10-cutoff', 'nitrous-oxide-ei310']

In [5]:
methods = [
     ('ecoinvent-3.10', 'ReCiPe 2016 v1.03, endpoint (H) no LT', 'total: human health no LT', 'human health no LT'),
#  ('ecoinvent-3.10', 'ReCiPe 2016 v1.03, endpoint (H) no LT', 'total: ecosystem quality no LT', 'ecosystem quality no LT'),
#  ('ecoinvent-3.10', 'ReCiPe 2016 v1.03, endpoint (H) no LT', 'total: natural resources no LT', 'natural resources no LT')
]
method_adj = 'human_health'
# method_adj = 'ecosystem_quality'
# method_adj = 'natural_resources'

In [6]:
methods

[('ecoinvent-3.10',
  'ReCiPe 2016 v1.03, endpoint (H) no LT',
  'total: human health no LT',
  'human health no LT')]

In [7]:
db = bd.Database('nitrous-oxide-ei310')
acts = [act for act in db if 'hydrogen peroxide production' in act['name'] or 'nitrous oxide production' in act['name'] or 'phenol production' in act['name']]
len(acts)

14

In [8]:
eidb = bd.Database('ecoinvent-3.10-cutoff')
act = [a for a in eidb if 'phenol production, cumene oxidation' in a['name'] and 'phenol' in a['reference product'] and 'RoW' in a['location']][0]
acts = acts + [act]
len(acts)

15

In [9]:
mc_df = pd.DataFrame()

In [10]:
demands = [{act:1} for act in acts]
all_demands = {k: 1 for demand in demands for k in demand}
lca = bc.LCA(demand=all_demands, method=methods[0], use_distributions=True)
lca.lci()

C_matrices = {}
for method in methods:
    lca.switch_method(method)
    C_matrices[method] = lca.characterization_matrix.copy()
    
results = {
    act['name']: [] for act in acts
}

for _ in range(1000):
    next(lca)
    for index, demand in enumerate(demands):
        lca.lci({key.id: value for key, value in demand.items()})
        for method in methods:
            name = str(list(demand.keys())[0])
            name = name.replace("dict_keys([", "").replace("])", "").replace("'", "")
            name = name[:name.find(' (')]
            results[name].append(
                (C_matrices[method] * lca.inventory).sum()
            )

for key, value in results.items():
    mc_df[key] = pd.Series(value)

In [11]:
mc_df

,"hydrogen peroxide production, AO, green hydrogen","phenol production, nitrous oxide, fossil","nitrous oxide production, BAU, fossil ammonia","hydrogen peroxide production, AO, fossil hydrogen","nitrous oxide production, AON 90 cryogenic, fossil ammonia","nitrous oxide production, AON 90 membrane, green ammonia","phenol production, hydrogen peroxide, fossil","phenol production, hydrogen peroxide, green","nitrous oxide production, AON 90 cryogenic, green ammonia","phenol production, nitrous oxide, green","nitrous oxide production, AON 100, green ammonia","nitrous oxide production, AON 90 membrane, fossil ammonia","nitrous oxide production, AON 100, fossil ammonia","nitrous oxide production, BAU, green ammonia","phenol production, cumene oxidation"
0,9.464258e-07,0.000006,0.000010,0.000002,0.000005,0.000003,0.000005,0.000005,0.000006,0.000006,0.000002,0.000003,0.000002,0.000009,0.000010
1,7.773784e-07,0.000006,0.000012,0.000001,0.000009,0.000002,0.000006,0.000005,0.000005,0.000005,0.000002,0.000007,0.000005,0.000008,0.000010
2,9.344023e-07,0.000008,0.000011,0.000002,0.000010,0.000002,0.000005,0.000006,0.000005,0.000007,0.000002,0.000005,0.000004,0.000008,0.000008
3,1.741668e-06,0.000008,0.000007,0.000002,0.000005,0.000005,0.000006,0.000008,0.000009,0.000009,0.000005,0.000002,0.000002,0.000011,0.000015
4,6.833320e-07,0.000011,0.000018,0.000002,0.000020,0.000002,0.000005,0.000005,0.000005,0.000006,0.000002,0.000018,0.000014,0.000008,0.000010
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
995,1.136724e-06,0.000011,0.000017,0.000002,0.000014,0.000003,0.000007,0.000006,0.000007,0.000008,0.000003,0.000011,0.000011,0.000009,0.000011
996,8.337367e-07,0.000008,0.000012,0.000002,0.000013,0.000003,0.000005,0.000005,0.000005,0.000006,0.000002,0.000010,0.000008,0.000008,0.000007
997,2.654075e-06,0.000013,0.000015,0.000002,0.000009,0.000006,0.000010,0.000009,0.000011,0.000011,0.000007,0.000008,0.000008,0.000013,0.000017
998,6.275289e-07,0.000013,0.000021,0.000001,0.000023,0.000002,0.000006,0.000005,0.000005,0.000006,0.000002,0.000019,0.000016,0.000008,0.000006


In [12]:
mc_df.to_excel(os.path.join('..', 'results', method_adj + '_monte_carlo.xlsx'))